In [49]:
import torch
print("CUDA beschikbaar:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA beschikbaar: True
Device: Tesla T4


In [50]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

In [51]:
URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(URL)
df = df.dropna(subset=["smiles"]).reset_index(drop=True)

print(f"Aantal moleculen: {len(df)}")
print(f"Class balance:\n{df['p_np'].value_counts()}")

Aantal moleculen: 2050
Class balance:
p_np
1    1567
0     483
Name: count, dtype: int64


In [52]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
from sklearn.model_selection import GroupKFold

def get_scaffold(smi):
    """Bereken Bemis-Murcko scaffold van een SMILES-string."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    scaffold_mol = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold_mol)

smiles = df["smiles"].tolist()
labels = df["p_np"].astype(int).tolist()

# --- Bereken scaffolds voor alle moleculen ---
print("Berekenen scaffolds...")
scaffolds = [get_scaffold(s) for s in smiles]
print(f"Unieke scaffolds: {len(set(scaffolds))} (uit {len(smiles)} moleculen)")

# Groepeer molecule-indices per scaffold
scaffold_to_indices = defaultdict(list)
for i, sc in enumerate(scaffolds):
    scaffold_to_indices[sc].append(i)

# Sorteer scaffold-groepen van groot naar klein
scaffold_groups = sorted(scaffold_to_indices.values(), key=lambda g: -len(g))

# --- Verdeel scaffolds over train+val en test, met class-balance-bewaking ---
n_total = len(smiles)
test_size_target = int(0.10 * n_total)
overall_pos_rate = np.mean(labels)   # ~0.76 voor BBBP

trainval_idx, test_idx = [], []
trainval_labels_so_far = []
test_labels_so_far     = []

for group in scaffold_groups:
    group_labels = [labels[i] for i in group]

    if len(test_idx) < test_size_target:
        # Kijk of test deze groep aankan zonder de balans te veel te verstoren
        candidate_test = test_labels_so_far + group_labels
        test_pos_rate = np.mean(candidate_test)

        if (abs(test_pos_rate - overall_pos_rate) < 0.10
                or len(test_idx) < 0.5 * test_size_target):
            test_idx.extend(group)
            test_labels_so_far.extend(group_labels)
            continue

    trainval_idx.extend(group)
    trainval_labels_so_far.extend(group_labels)

# --- Maak de bijbehorende lijsten ---
trainval_smiles    = [smiles[i]    for i in trainval_idx]
trainval_labels    = [labels[i]    for i in trainval_idx]
trainval_scaffolds = [scaffolds[i] for i in trainval_idx]  # nodig voor CV
test_smiles        = [smiles[i]    for i in test_idx]
test_labels        = [labels[i]    for i in test_idx]

# --- Rapporteer ---
print(f"\nTrain+Val: {len(trainval_smiles)} | Test: {len(test_smiles)}")
print(f"Class balance train+val: {np.bincount(trainval_labels)} "
      f"(pos rate: {np.mean(trainval_labels):.3f})")
print(f"Class balance test:      {np.bincount(test_labels)} "
      f"(pos rate: {np.mean(test_labels):.3f})")
print(f"Overall pos rate:        {overall_pos_rate:.3f}")

# --- Sanity check op scaffold-disjunctie ---
trainval_scaffold_set = set(trainval_scaffolds)
test_scaffold_set     = {scaffolds[i] for i in test_idx}
overlap = trainval_scaffold_set & test_scaffold_set
print(f"\nScaffold overlap train+val ↔ test: {len(overlap)} "
      f"(zou 0 moeten zijn)")

Berekenen scaffolds...


[10:35:10] Explicit valence for atom # 1 N, 4, is greater than permitted
[10:35:10] WARNING: not removing hydrogen atom without neighbors
[10:35:10] Explicit valence for atom # 6 N, 4, is greater than permitted
[10:35:10] WARNING: not removing hydrogen atom without neighbors
[10:35:10] WARNING: not removing hydrogen atom without neighbors
[10:35:10] WARNING: not removing hydrogen atom without neighbors
[10:35:10] WARNING: not removing hydrogen atom without neighbors
[10:35:10] WARNING: not removing hydrogen atom without neighbors
[10:35:10] WARNING: not removing hydrogen atom without neighbors
[10:35:10] Explicit valence for atom # 6 N, 4, is greater than permitted
[10:35:10] WARNING: not removing hydrogen atom without neighbors
[10:35:10] WARNING: not removing hydrogen atom without neighbors
[10:35:11] WARNING: not removing hydrogen atom without neighbors
[10:35:11] WARNING: not removing hydrogen atom without neighbors
[10:35:11] Explicit valence for atom # 11 N, 4, is greater than pe

Unieke scaffolds: 1102 (uit 2050 moleculen)

Train+Val: 1803 | Test: 247
Class balance train+val: [ 435 1368] (pos rate: 0.759)
Class balance test:      [ 48 199] (pos rate: 0.806)
Overall pos rate:        0.764

Scaffold overlap train+val ↔ test: 0 (zou 0 moeten zijn)


[10:35:11] WARNING: not removing hydrogen atom without neighbors
[10:35:11] WARNING: not removing hydrogen atom without neighbors


In [53]:
all_chars = set()
for smi in trainval_smiles:   # was: train_smiles
    all_chars.update(smi)

sorted_chars = sorted(all_chars)
char_to_idx = {"<PAD>": 0, "<UNK>": 1}
for i, c in enumerate(sorted_chars, start=2):
    char_to_idx[c] = i

idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(char_to_idx)
print(f"Vocab size: {vocab_size}")

Vocab size: 41


In [54]:
lengths = [len(s) for s in train_smiles]
print(f"SMILES lengtes — min: {min(lengths)}, max: {max(lengths)}, "
      f"mean: {np.mean(lengths):.1f}, 95-percentiel: {int(np.percentile(lengths, 95))}")

SMILES lengtes — min: 3, max: 400, mean: 51.8, 95-percentiel: 106


In [55]:
MAX_LENGTH = 200

def encode_smiles(smi, char_to_idx, max_length=MAX_LENGTH):
    """Zet een SMILES-string om in een lijst integers van vaste lengte."""
    # Truncate als hij te lang is
    smi = smi[:max_length]
    # Map elk character naar zijn index, onbekende → <UNK>
    ids = [char_to_idx.get(c, char_to_idx["<UNK>"]) for c in smi]
    # Pad rechts met 0 (=<PAD>) tot max_length
    ids = ids + [char_to_idx["<PAD>"]] * (max_length - len(ids))
    return ids


class SMILES_CNN_Dataset(Dataset):
    def __init__(self, smiles, labels, char_to_idx, max_length=MAX_LENGTH):
        self.smiles = smiles
        self.labels = labels
        self.char_to_idx = char_to_idx
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        ids = encode_smiles(self.smiles[idx], self.char_to_idx, self.max_length)
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_ds = SMILES_CNN_Dataset(train_smiles, train_labels, char_to_idx)
val_ds   = SMILES_CNN_Dataset(val_smiles,   val_labels,   char_to_idx)
test_ds  = SMILES_CNN_Dataset(test_smiles,  test_labels,  char_to_idx)

# Even checken
print(f"Train dataset size: {len(train_ds)}")
print(f"Eerste sample shape: {train_ds[0]['input_ids'].shape}")
print(f"Eerste 30 tokens van eerste sample: {train_ds[0]['input_ids'][:30]}")

Train dataset size: 1660
Eerste sample shape: torch.Size([200])
Eerste 30 tokens van eerste sample: tensor([35, 11, 35, 35, 35, 12, 35,  4, 35,  4, 23,  4, 27, 31, 23, 21, 21, 25,
        33,  4, 23, 23,  5, 35, 13, 35, 35, 35, 35, 35])


In [56]:
class SMILES_CNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=64,
                 kernel_sizes=(3, 5, 7), dropout=0.5, num_classes=2,
                 class_weights=None):   # NIEUW
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

        # NIEUW: bewaar class weights als buffer (gaat automatisch mee naar GPU)
        if class_weights is not None:
            self.register_buffer(
                "class_weights",
                torch.tensor(class_weights, dtype=torch.float32)
            )
        else:
            self.class_weights = None

    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        x = x.permute(0, 2, 1)

        conv_outputs = []
        for conv in self.convs:
            c = F.relu(conv(x))
            p = F.max_pool1d(c, c.size(2)).squeeze(2)
            conv_outputs.append(p)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        logits = self.fc(x)

        loss = None
        if labels is not None:
            # NIEUW: weight= toegevoegd
            loss = F.cross_entropy(logits, labels, weight=self.class_weights)

        return {"loss": loss, "logits": logits}

In [57]:
from torch.utils.data import DataLoader

# DataLoaders: zorgen voor batching en (voor train) shuffling
BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)


def evaluate(model, loader, device):
    """Evalueer model op een loader, geeft loss/accuracy/AUC terug."""
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            out = model(input_ids, labels=labels)
            total_loss += out["loss"].item() * input_ids.size(0)
            all_logits.append(out["logits"].cpu())
            all_labels.append(labels.cpu())

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels).numpy()
    preds = logits.argmax(dim=1).numpy()
    probs = torch.softmax(logits, dim=1)[:, 1].numpy()

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(labels, preds),
        "roc_auc":  roc_auc_score(labels, probs),
    }


def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, weight_decay=1e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_auc = 0.0
    best_state = None
    history = []

    print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>9} | {'Val Acc':>7} | {'Val AUC':>7}")
    print("-" * 55)

    for epoch in range(1, epochs + 1):
        # ---- Training ----
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            out = model(input_ids, labels=labels)
            out["loss"].backward()
            optimizer.step()

            epoch_loss += out["loss"].item() * input_ids.size(0)

        train_loss = epoch_loss / len(train_loader.dataset)

        # ---- Validation ----
        val_metrics = evaluate(model, val_loader, device)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics,
        })

        print(f"{epoch:>5} | {train_loss:>10.4f} | {val_metrics['loss']:>9.4f} "
              f"| {val_metrics['accuracy']:>7.4f} | {val_metrics['roc_auc']:>7.4f}")

        # Sla het beste model op (op basis van val AUC)
        if val_metrics["roc_auc"] > best_val_auc:
            best_val_auc = val_metrics["roc_auc"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Laad het beste model terug
    model.load_state_dict(best_state)
    print(f"\nBeste val AUC: {best_val_auc:.4f}")
    return history

In [58]:
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight

N_FOLDS = 5
EPOCHS = 30
BATCH_SIZE = 32

torch.manual_seed(42)
np.random.seed(42)

# GroupKFold: zorgt dat scaffolds NIET in train én val van dezelfde fold zitten
gkf = GroupKFold(n_splits=N_FOLDS)

fold_results = []
fold_models = []

# Verzamel val-probabilities per fold — nodig voor threshold tuning later
all_val_probs  = []
all_val_labels = []

trainval_smiles_arr = np.array(trainval_smiles)
trainval_labels_arr = np.array(trainval_labels)

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(trainval_smiles_arr, trainval_labels_arr, groups=trainval_scaffolds),
    start=1
):
    print(f"\n========== FOLD {fold}/{N_FOLDS} ==========")

    fold_train_smiles = trainval_smiles_arr[train_idx].tolist()
    fold_train_labels = trainval_labels_arr[train_idx].tolist()
    fold_val_smiles   = trainval_smiles_arr[val_idx].tolist()
    fold_val_labels   = trainval_labels_arr[val_idx].tolist()

    print(f"Train class balance: {np.bincount(fold_train_labels)}")
    print(f"Val class balance:   {np.bincount(fold_val_labels)}")

    # Class weights per fold (op trainportie)
    cw = compute_class_weight(
        "balanced",
        classes=np.array([0, 1]),
        y=np.array(fold_train_labels)
    )
    print(f"Class weights: {cw}")

    # Datasets + loaders
    fold_train_ds = SMILES_CNN_Dataset(fold_train_smiles, fold_train_labels, char_to_idx)
    fold_val_ds   = SMILES_CNN_Dataset(fold_val_smiles,   fold_val_labels,   char_to_idx)
    fold_train_loader = DataLoader(fold_train_ds, batch_size=BATCH_SIZE, shuffle=True)
    fold_val_loader   = DataLoader(fold_val_ds,   batch_size=BATCH_SIZE, shuffle=False)

    # Trainen
    fold_model = SMILES_CNN(vocab_size=vocab_size, class_weights=cw).to(device)
    history = train_model(fold_model, fold_train_loader, fold_val_loader, epochs=EPOCHS)

    best_val = max(history, key=lambda h: h["roc_auc"])
    fold_results.append(best_val)
    fold_models.append(fold_model)

    # --- NIEUW: bewaar val-predictions van het beste model ---
    # train_model heeft het beste model state al teruggeladen
    fold_model.eval()
    fold_val_probs, fold_val_lbls = [], []
    with torch.no_grad():
        for batch in fold_val_loader:
            out = fold_model(batch["input_ids"].to(device))
            probs = torch.softmax(out["logits"], dim=1)[:, 1].cpu().numpy()
            fold_val_probs.append(probs)
            fold_val_lbls.append(batch["labels"].numpy())
    all_val_probs.append(np.concatenate(fold_val_probs))
    all_val_labels.append(np.concatenate(fold_val_lbls))

# --- Samenvatting ---
print("\n========== CV RESULTATEN (scaffold split) ==========")
val_aucs = [r["roc_auc"] for r in fold_results]
val_accs = [r["accuracy"] for r in fold_results]
print(f"Val ROC-AUC per fold: {[f'{a:.4f}' for a in val_aucs]}")
print(f"Mean ± std ROC-AUC:   {np.mean(val_aucs):.4f} ± {np.std(val_aucs):.4f}")
print(f"Mean ± std Accuracy:  {np.mean(val_accs):.4f} ± {np.std(val_accs):.4f}")


========== FOLD 1/5 ==========
Train class balance: [ 352 1090]
Val class balance:   [ 83 278]
Class weights: [2.04829545 0.66146789]
Epoch | Train Loss |  Val Loss | Val Acc | Val AUC
-------------------------------------------------------
    1 |     0.5151 |    0.3922 |  0.7673 |  0.9085
    2 |     0.3676 |    0.3389 |  0.8227 |  0.9116
    3 |     0.3832 |    0.4487 |  0.8532 |  0.9161
    4 |     0.3787 |    0.3593 |  0.8144 |  0.9191
    5 |     0.2938 |    0.3424 |  0.8393 |  0.9145
    6 |     0.2944 |    0.3645 |  0.8061 |  0.9138
    7 |     0.2507 |    0.3472 |  0.8144 |  0.9164
    8 |     0.2082 |    0.3184 |  0.8393 |  0.9247
    9 |     0.1863 |    0.3232 |  0.8283 |  0.9222
   10 |     0.2393 |    0.3230 |  0.8338 |  0.9251
   11 |     0.1805 |    0.3168 |  0.8560 |  0.9279
   12 |     0.1823 |    0.3309 |  0.8532 |  0.9249
   13 |     0.1672 |    0.3293 |  0.8393 |  0.9262
   14 |     0.1415 |    0.3414 |  0.8421 |  0.9269
   15 |     0.1444 |    0.3497 |  0.8421 |  

In [59]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, f1_score, average_precision_score
)

# === Stap 1: vind optimale threshold op out-of-fold val data ===
# Belangrijk: NIET op test set zoeken — anders is het cherry-picking
oof_probs  = np.concatenate(all_val_probs)
oof_labels = np.concatenate(all_val_labels)

thresholds = np.linspace(0.05, 0.95, 91)
val_f1s = [f1_score(oof_labels, (oof_probs > t).astype(int), average="macro")
           for t in thresholds]
best_t = thresholds[int(np.argmax(val_f1s))]
print(f"Optimale threshold (op out-of-fold val): {best_t:.3f}")
print(f"Macro F1 op val bij deze threshold:      {max(val_f1s):.4f}")

# === Stap 2: ensemble-voorspelling op test set ===
test_ds = SMILES_CNN_Dataset(test_smiles, test_labels, char_to_idx)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

all_probs_per_fold = []
for m in fold_models:
    m.eval()
    probs = []
    with torch.no_grad():
        for batch in test_loader:
            out = m(batch["input_ids"].to(device))
            probs.append(torch.softmax(out["logits"], dim=1)[:, 1].cpu().numpy())
    all_probs_per_fold.append(np.concatenate(probs))

mean_probs = np.mean(all_probs_per_fold, axis=0)
y_true = np.array(test_labels)

# === Stap 3: rapporteer met DEFAULT threshold (0.5) ===
y_pred_default = (mean_probs > 0.5).astype(int)
print("\n=== Test resultaten — default threshold (0.5) ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred_default):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_true, mean_probs):.4f}")
print(f"PR-AUC:   {average_precision_score(y_true, mean_probs):.4f}")
print(f"\nConfusion matrix:\n{confusion_matrix(y_true, y_pred_default)}")
print(f"\n{classification_report(y_true, y_pred_default, target_names=['geen BBB', 'wel BBB'])}")

# === Stap 4: rapporteer met TUNED threshold ===
y_pred_tuned = (mean_probs > best_t).astype(int)
print(f"\n=== Test resultaten — tuned threshold ({best_t:.3f}) ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred_tuned):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_true, mean_probs):.4f}  (threshold-onafhankelijk)")
print(f"PR-AUC:   {average_precision_score(y_true, mean_probs):.4f}  (threshold-onafhankelijk)")
print(f"\nConfusion matrix:\n{confusion_matrix(y_true, y_pred_tuned)}")
print(f"\n{classification_report(y_true, y_pred_tuned, target_names=['geen BBB', 'wel BBB'])}")

Optimale threshold (op out-of-fold val): 0.450
Macro F1 op val bij deze threshold:      0.8481

=== Test resultaten — default threshold (0.5) ===
Accuracy: 0.7611
ROC-AUC:  0.9427
PR-AUC:   0.9864

Confusion matrix:
[[ 47   1]
 [ 58 141]]

              precision    recall  f1-score   support

    geen BBB       0.45      0.98      0.61        48
     wel BBB       0.99      0.71      0.83       199

    accuracy                           0.76       247
   macro avg       0.72      0.84      0.72       247
weighted avg       0.89      0.76      0.79       247


=== Test resultaten — tuned threshold (0.450) ===
Accuracy: 0.7935
ROC-AUC:  0.9427  (threshold-onafhankelijk)
PR-AUC:   0.9864  (threshold-onafhankelijk)

Confusion matrix:
[[ 47   1]
 [ 50 149]]

              precision    recall  f1-score   support

    geen BBB       0.48      0.98      0.65        48
     wel BBB       0.99      0.75      0.85       199

    accuracy                           0.79       247
   macro avg    